In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn import metrics

In [2]:
delhi = pd.read_csv("delhi_rent.csv")
mumbai = pd.read_csv("mumbai_rent.csv")
gurgaon = pd.read_csv("gurgaon_rent.csv")

In [3]:
print("Delhi columns:")
print(delhi.columns)

print("\nMumbai columns:")
print(mumbai.columns)

print("\nGurgaon columns:")
print(gurgaon.columns)

Delhi columns:
Index(['Unnamed: 0', 'size_sq_ft', 'propertyType', 'bedrooms', 'latitude',
       'longitude', 'localityName', 'suburbName', 'cityName', 'price',
       'companyName', 'closest_mtero_station_km', 'AP_dist_km',
       'Aiims_dist_km', 'NDRLW_dist_km'],
      dtype='object')

Mumbai columns:
Index(['Title', 'Price', 'Area', 'BHK', 'Bathrooms', 'Furnished', 'Balconies',
       'Floor', 'Ownership', 'Facing', 'Amenities', 'Transaction Type',
       'Property Type', 'Location', 'Year of Construction', 'Is Luxury',
       'Description', 'Property Image'],
      dtype='object')

Gurgaon columns:
Index(['Price', 'Status', 'Area', 'Rate per sqft', 'Property Type', 'Locality',
       'Builder Name', 'RERA Approval', 'BHK_Count', 'Socity', 'Company Name',
       'Flat Type'],
      dtype='object')


In [4]:
print(delhi.shape)
print(mumbai.shape)
print(gurgaon.shape)

(17890, 15)
(23405, 18)
(19515, 12)


In [5]:
delhi = delhi.rename(columns={
    "size_sq_ft": "size_sqft",
    "propertyType": "property_type",
    "bedrooms": "bhk",
    "localityName": "locality",
    "cityName": "city",
    "price": "rent"
})

delhi = delhi[
    ["city","locality","property_type","bhk","size_sqft","latitude","longitude","rent"]
]

In [6]:
mumbai = mumbai.rename(columns={
    "Area": "size_sqft",
    "BHK": "bhk",
    "Bathrooms": "bathrooms",
    "Balconies": "balcony",
    "Furnished": "furnishing",
    "Property Type": "property_type",
    "Location": "locality",
    "Price": "rent"
})

mumbai["city"] = "Mumbai"

In [7]:
gurgaon = gurgaon.rename(columns={
    "Area": "size_sqft",
    "BHK_Count": "bhk",
    "Property Type": "property_type",
    "Locality": "locality",
    "Price": "rent"
})

gurgaon["city"] = "Gurgaon"

In [8]:
columns = ["city","locality","property_type","bhk","size_sqft","rent"]

delhi = delhi[columns]
mumbai = mumbai[columns]
gurgaon = gurgaon[columns]

In [9]:
df = pd.concat([delhi, mumbai, gurgaon], ignore_index=True)

In [10]:
df.shape

(60810, 6)

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60810 entries, 0 to 60809
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   city           60810 non-null  object
 1   locality       60810 non-null  object
 2   property_type  60810 non-null  object
 3   bhk            60399 non-null  object
 4   size_sqft      60810 non-null  object
 5   rent           60808 non-null  object
dtypes: object(6)
memory usage: 2.8+ MB


In [12]:
df = df.dropna()

In [13]:
df.shape

(60399, 6)

In [15]:
df["bhk"] = (
    df["bhk"]
    .astype(str)
    .str.extract(r'(\d+)')
    .astype(int)
)

In [16]:
df["size_sqft"] = (
    df["size_sqft"]
    .astype(str)
    .str.replace(",", "")
    .str.extract(r'(\d+)')
    .astype(float)
)

In [17]:
df["rent"] = (
    df["rent"]
    .astype(str)
    .str.replace(",", "")
    .str.replace("₹", "")
    .str.extract(r'(\d+)')
    .astype(float)
)

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 60399 entries, 0 to 60809
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   city           60399 non-null  object 
 1   locality       60399 non-null  object 
 2   property_type  60399 non-null  object 
 3   bhk            60399 non-null  int64  
 4   size_sqft      59475 non-null  float64
 5   rent           60314 non-null  float64
dtypes: float64(2), int64(1), object(3)
memory usage: 3.2+ MB


In [19]:
df = df.dropna()

In [20]:
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
Index: 59433 entries, 0 to 60809
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   city           59433 non-null  object 
 1   locality       59433 non-null  object 
 2   property_type  59433 non-null  object 
 3   bhk            59433 non-null  int64  
 4   size_sqft      59433 non-null  float64
 5   rent           59433 non-null  float64
dtypes: float64(2), int64(1), object(3)
memory usage: 3.2+ MB


(59433, 6)

In [21]:
df.describe()

,bhk,size_sqft,rent
count,59433.000000,59433.000000,5.943300e+04
mean,2.520502,1644.886006,1.325105e+07
std,2.970129,5552.204300,3.900469e+07
min,0.000000,2.000000,1.000000e+00
25%,2.000000,815.000000,1.600000e+04
50%,2.000000,1251.000000,4.800000e+04
75%,3.000000,1955.000000,1.400000e+07
max,132.000000,958320.000000,1.226300e+09


In [22]:
df["city"].unique()
df["property_type"].unique()
df["locality"].nunique()

1013

In [23]:
df = df[
    (df["bhk"] > 0) &
    (df["bhk"] <= 6) &
    (df["size_sqft"] >= 300) &
    (df["size_sqft"] <= 8000) &
    (df["rent"] >= 5000) &
    (df["rent"] <= 300000)
]

In [24]:
df.shape

(32034, 6)

In [25]:
df = pd.get_dummies(df, drop_first=True)

In [28]:
df.isnull().sum()

,0
bhk,0
size_sqft,0
rent,0
city_Mumbai,0
locality_100 Feet Road,0
...,...
property_type_Independent House,0
property_type_Penthouse,0
property_type_Residential House,0
property_type_Service Apartment,0


In [29]:
df.describe()

,bhk,size_sqft,rent
count,32034.000000,32034.000000,32034.000000
mean,2.001904,1050.700162,39586.737779
std,0.845194,597.303754,29055.985355
min,1.000000,300.000000,5000.000000
25%,1.000000,650.000000,18000.000000
50%,2.000000,900.000000,32000.000000
75%,3.000000,1250.000000,55000.000000
max,6.000000,8000.000000,300000.000000


In [31]:
df.columns

Index(['bhk', 'size_sqft', 'rent', 'city_Mumbai', 'locality_100 Feet Road',
       'locality_40 Feet Road', 'locality_45 Sector 21 Road',
       'locality_48 Sector 22 Road', 'locality_52 Sector Road',
       'locality_60 Feet Road',
       ...
       'locality_roop villa apartment delhi dwarka sector 19',
       'locality_south delhi apartment sector 4', 'locality_vikaspuri',
       'property_type_Builder Floor Apartment',
       'property_type_Independent Floor', 'property_type_Independent House',
       'property_type_Penthouse', 'property_type_Residential House',
       'property_type_Service Apartment', 'property_type_Villa'],
      dtype='object', length=762)

In [34]:
[col for col in df.columns if "city" in col]

['city_Mumbai']

In [35]:
df.filter(like="locality_").columns[:20]

Index(['locality_100 Feet Road', 'locality_40 Feet Road',
       'locality_45 Sector 21 Road', 'locality_48 Sector 22 Road',
       'locality_52 Sector Road', 'locality_60 Feet Road',
       'locality_72 Sector 23 Road', 'locality_75 Noida Road',
       'locality_75 Sector 22 Road', 'locality_76 Noida Road',
       'locality_94 B Block Road', 'locality_944 B Block Road',
       'locality_A 2 Block', 'locality_A 3 Block', 'locality_A 6 Block',
       'locality_A1 Block Paschim Vihar Delhi',
       'locality_A4 Block Paschim Vihar', 'locality_A5 Block',
       'locality_AC Block Shalimar Bagh', 'locality_AD Block Pitampura'],
      dtype='object')

In [36]:
df.shape

(32034, 762)

# MODEL TRAINING AND SPLITTING

In [38]:
y = df["rent"]

In [39]:
X = df.drop("rent", axis=1)

In [40]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [41]:
print(X_train.shape)
print(X_test.shape)

(25627, 761)
(6407, 761)


In [42]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)

In [43]:
y_pred = model.predict(X_test)

In [44]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("R2 Score:", r2)

MAE: 9196.785114006429
R2 Score: 0.7304294894654813


In [45]:
RandomForestRegressor(
    n_estimators=400,
    random_state=42,
    n_jobs=-1
)

RandomForestRegressor(n_estimators=400, n_jobs=-1, random_state=42)

In [46]:
RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

RandomForestRegressor(max_depth=20, n_estimators=300, n_jobs=-1,
                      random_state=42)

In [47]:
from sklearn.ensemble import GradientBoostingRegressor

In [48]:
import pickle

with open("rent_prediction_model.pkl", "wb") as f:
    pickle.dump(model, f)

In [49]:
with open("rent_model_features.pkl", "wb") as f:
    pickle.dump(X.columns, f)

In [50]:
from google.colab import files

files.download("rent_prediction_model.pkl")
files.download("rent_model_features.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>